In [ ]:
# Ensure jacopy is importable when this notebook is opened
# directly (not via pytest). Walks up from the notebook's
# directory to the repo root and prepends it to sys.path if
# jacopy isn't already installed into this kernel.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

# 20, Connection, curvature, and Bianchi identities

Companion notebook to [20_connection_curvature.md](20_connection_curvature.md). `AffineConnection` carries `∇_X Y`; `Torsion` / `Curvature` are the structural obstructions; `BianchiProblem` bundles every rule needed to close the two Bianchi identities mechanically.

## `AffineConnection` and `∇_X Y`

An atom + an `eval(X, Y)` builder. Four engine rules realise X-linearity, X-scalar-pull, Y-additivity, and Y-Leibniz.

In [ ]:
from jacopy.algebra.derivation import Derivation
from jacopy.calculus.connection import AffineConnection

nabla = AffineConnection("∇")
X, Y, W, Z = (Derivation(s, 0) for s in ("X", "Y", "W", "Z"))

print(f"connection : {nabla}")
print(f"∇_X Y      : {nabla.eval(X, Y)}")

## Torsion and curvature

$$T(X, Y) := \nabla_X Y - \nabla_Y X - [X, Y]$$
$$R(X, Y) Z := \nabla_X \nabla_Y Z - \nabla_Y \nabla_X Z - \nabla_{[X,Y]} Z$$

Both inert until `TorsionDefinitionDefinition` / `CurvatureDefinitionDefinition` fire.

In [ ]:
from jacopy.calculus.torsion_curvature import Torsion, Curvature

T = Torsion(nabla, X, Y)
R = Curvature(nabla, X, Y, W)
print(f"T(X, Y)   : {T}")
print(f"R(X, Y) W : {R}")

## `BianchiProblem`, the wrapper

Bundles every rule needed: four connection axioms, torsion / curvature definitions, covariant-derivative definitions, and the LBVF (or `BracketApply`) closure family. The `registry` powers the Y-Leibniz `Graded(degree=0)` test on function factors.

In [ ]:
from jacopy.core.registry import PropertyRegistry
from jacopy.library.bianchi_problem import BianchiProblem

reg = PropertyRegistry()
prob = BianchiProblem(nabla, registry=reg)
print(f"engine rules : {len(prob.engine.definitions)}")
print(f"connection   : {prob.connection}")

## First Bianchi identity

$$\mathrm{cycl}_{X,Y,Z}\, R(X, Y) Z = \mathrm{cycl}_{X,Y,Z}\, [(\nabla_X T)(Y, Z) + T(T(X, Y), Z)]$$

`prove_first_bianchi` builds the difference of both sides, expands through the engine, and reports `ok=True` iff the residue collapses to `0`.

In [ ]:
res1 = prob.prove_first_bianchi(X, Y, W)
print(f"Bianchi I : ok={res1.ok}, steps={len(res1.lhs_steps)}")

## Second Bianchi identity

$$\mathrm{cycl}_{X,Y,Z}\, (\nabla_X R)(Y, Z) W = \mathrm{cycl}_{X,Y,Z}\, R(X, T(Y, Z)) W$$

Same protocol, one extra fixed argument.

In [ ]:
res2 = prob.prove_second_bianchi(X, Y, W, Z)
print(f"Bianchi II : ok={res2.ok}, steps={len(res2.lhs_steps)}")

## Koszul connection, same identities on `T*M`

`koszul_connection(name, *, anchor)` produces an algebroid connection on `T*M` for Poisson problems. Same `BianchiProblem` wrapper works, engine swaps in `BracketApply` closure rules and routes function-action through the anchor.

In [ ]:
from jacopy.calculus.connection import koszul_connection
from jacopy.calculus.anchor import Anchor

pi_sharp = Anchor("π^♯")
nabla_tilde = koszul_connection("∇̃", anchor=pi_sharp)
print(f"connection : {nabla_tilde}")
print(f"anchor     : {nabla_tilde.anchor}")
print(f"bracket    : {nabla_tilde.bracket}")

## Summary

* `AffineConnection(name)` atom + `eval(X, Y)` builder + four defining-axiom engine rules.
* `Torsion(∇, X, Y)` / `Curvature(∇, X, Y, Z)` inert until their definitions fire.
* `BianchiProblem` bundles every rule; `prove_first_bianchi` / `prove_second_bianchi` close in ~60-63 steps.
* `koszul_connection` for the cotangent variant, same wrapper, `BracketApply` closure family swapped in.